In [0]:
# 1. Crear Catálogo electrocasa y esquemas Medallion
spark.sql("CREATE CATALOG IF NOT EXISTS electrocasa")
spark.sql("USE CATALOG electrocasa")

for schema_name in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

# 2. Crear Volúmenes en el esquema bronze
spark.sql("CREATE VOLUME IF NOT EXISTS electrocasa.bronze.landing_volume")
spark.sql("CREATE VOLUME IF NOT EXISTS electrocasa.bronze.checkpoints")

print("Catálogo 'electrocasa', schemas (bronze, silver, gold) y volúmenes creados exitosamente.")

In [0]:
# COMMAND ----------
# 3. Creación de Grupos y Concesión de Permisos (Unity Catalog)

# En Databricks SQL estándar la sintaxis es: CREATE GROUP <nombre> (sin IF NOT EXISTS).
# En Databricks Free Edition los grupos se gestionan a nivel de cuenta (Account Console).
grupos = ["ingenieria", "analistas", "auditoria"]

for group in grupos:
    try:
        spark.sql(f"CREATE GROUP {group}")
        print(f"Grupo '{group}' creado con éxito.")
    except Exception as e:
        print(f"Grupo '{group}' no se pudo crear por DDL (administrado a nivel de cuenta): {type(e).__name__}")

# Otorgar permisos: aplicamos a tu usuario activo para validar la ejecución real en Free Edition
current_user = spark.sql("SELECT current_user()").collect()[0][0]

try:
    spark.sql(f"GRANT ALL PRIVILEGES ON CATALOG electrocasa TO `{current_user}`")
    spark.sql(f"GRANT ALL PRIVILEGES ON SCHEMA electrocasa.gold TO `{current_user}`")
    print(f"Privilegios otorgados exitosamente al usuario actual ({current_user}).")
except Exception as e:
    print(f"Aviso al asignar privilegios: {e}")

# Documentación para evaluación del docente (Sintaxis formal de producción):
"""
-- Scripts DDL para ambiente Enterprise/Producción con Account Console configurado:
GRANT ALL PRIVILEGES ON CATALOG electrocasa TO `ingenieria`;
GRANT USAGE ON CATALOG electrocasa TO `analistas`, `auditoria`;
GRANT USAGE, SELECT ON SCHEMA electrocasa.gold TO `analistas`, `auditoria`;
"""

In [0]:
# 4. Registrar función de enmascaramiento dinámico para DNI/Salario de Empleados
spark.sql("""
CREATE OR REPLACE FUNCTION electrocasa.silver.mask_dni(dni STRING)
RETURNS STRING
RETURN IF(IS_ACCOUNT_GROUP_MEMBER('ingenieria'), dni, CONCAT('***-***-', RIGHT(dni, 2)))
""")

print("Función de enmascaramiento electrocasa.silver.mask_dni registrada.")

In [0]:
# COMMAND ----------
# 2.B Lakehouse Federation: Connection y Foreign Catalog (Azure SQL)
# Requerido por el pipeline declarativo
spark.sql("""
CREATE CONNECTION IF NOT EXISTS azure_sql_electrocasa
TYPE SQLSERVER
OPTIONS (
  host 'analyticsdmc.database.windows.net',
  port '1433',
  user 'sqladmin',
  password 'mdp123$$'
)
""")

spark.sql("""
CREATE FOREIGN CATALOG IF NOT EXISTS electrocasa_sql_source
USING CONNECTION azure_sql_electrocasa
OPTIONS (database 'electrocasadb')
""")

print("Connection 'azure_sql_electrocasa' y Foreign Catalog 'electrocasa_sql_source' listos.")